In [8]:
import sys
import os
import random
import torch
import numpy as np
from pathlib import Path
import wandb
import pandas as pd
import torchvision.transforms as transforms
from PIL import Image

In [9]:
sys.path.append(os.path.abspath("../"))

from importlib import reload

import Segmentation.Postprocessing.segmentation_postprocessing
import Segmentation.SAM.sam_lora
import Segmentation.SAM.sam_lora_util
import Segmentation.Util.env_utils
import Segmentation.Util.dataset_util

import JunctionDetection.SkeletonizeDetect.segmentation_junction_detection

reload(Segmentation.Postprocessing.segmentation_postprocessing)
reload(Segmentation.SAM.sam_lora)
reload(Segmentation.SAM.sam_lora_util)
reload(Segmentation.Util.env_utils)
reload(Segmentation.Util.dataset_util)
reload(JunctionDetection.SkeletonizeDetect.segmentation_junction_detection)

loaded environment variables from: /local/scratch/jhehli/ForkSight/Segmentation/.env
loaded environment variables from: /local/scratch/jhehli/ForkSight/Segmentation/.env


<module 'JunctionDetection.SkeletonizeDetect.segmentation_junction_detection' from '/local/scratch/jhehli/ForkSight/JunctionDetection/SkeletonizeDetect/segmentation_junction_detection.py'>

In [10]:
Segmentation.Util.env_utils.load_segmentation_env()

SEED = Segmentation.Util.env_utils.load_as("SEED", int, 42)

RAW_DATA_DIR = os.getenv("RAW_DATA_DIR")
MODEL_CHECKPOINTS_DIR = os.getenv("MODEL_CHECKPOINTS_DIR")
DATASETS_DIR = os.getenv("DATASETS_DIR")

HIGHRES_IMG_PATCHES_DIR_NAME = os.getenv(
    "HIGHRES_IMG_PATCHES_DIR_NAME", "img_patches_1024")
HIGHRES_MASK_PATCHES_DIR_NAME = os.getenv(
    "HIGHRES_MASK_PATCHES_DIR_NAME", "mask_patches_1024")
HIGHRES_IMG_DIR_NAME = os.getenv("HIGHRES_IMG_DIR_NAME", "images_4096")
HIGHRES_MASK_DIR_NAME = os.getenv("HIGHRES_MASK_DIR_NAME", "masks_4096")

WANDB_ENTITY = os.getenv("WANDB_ENTITY", "EM_IMCR_BIOVSION")
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "ForkSight-SAM")

DATASET_JUNCTION_COORDS_CVAT_XML_PATH = os.getenv(
    "DATASET_JUNCTION_COORDS_CVAT_XML_PATH", None)

if RAW_DATA_DIR is None or MODEL_CHECKPOINTS_DIR is None or DATASETS_DIR is None:
    raise ValueError(
        "RAW_DATA_DIR, MODEL_CHECKPOINTS_DIR or DATASETS_DIR environment variable not set.")

loaded environment variables from: /local/scratch/jhehli/ForkSight/Segmentation/.env


In [11]:
loss_config_keys = ["bce_loss_weight", "cl_dice_loss_weight", "dice_loss_weight", "focal_loss_weight",
                    "junction_heatmap_weight_scale", "junction_patch_weight", "skeleton_recall_loss_weight", "topological_loss_weight"]

In [7]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [12]:
def load_transform_image(path: Path, is_mask: bool = False):
    transform = transforms.Compose([transforms.ToTensor(),
                                    transforms.Lambda(lambda t: t.repeat(3, 1, 1) if t.shape[0] == 1 and not is_mask else t)])
    img = Image.open(path)
    return transform(img)


def get_single_image_input_list(image: torch.Tensor):
    return [{
        "image": img,
        "original_size": (img.shape[1], img.shape[2])
    } for img in image.unsqueeze(0).unbind(0)]

In [13]:
print("cuda available: ", torch.cuda.is_available())
!nvidia-smi --query-gpu=index,utilization.gpu --format=csv

device_idx = int(input("Enter the index of the cuda device: "))
device = torch.device(f"cuda:{device_idx}" if torch.cuda.is_available() else "cpu")

print(f"\nUsing device: {device}")

cuda available:  True
index, utilization.gpu [%]
0, 0 %
1, 0 %
2, 0 %
3, 0 %
4, 0 %
5, 0 %
6, 0 %
7, 0 %

Using device: cuda:0


In [ ]:
params_artifact_name = "params_minloss"

models = [
    "SAM_LoRA_Finetuning_20260212_155155",
    "SAM_LoRA_Finetuning_20260212_155412",
    "SAM_LoRA_Finetuning_20260212_155506",
    "SAM_LoRA_Finetuning_20260212_155528",
    "SAM_LoRA_Finetuning_20260212_155556",
    "SAM_LoRA_Finetuning_20260212_161635",
    "SAM_LoRA_Finetuning_20260212_163023",
]

api = wandb.Api()

results = {}

runs = runs = [run for run in list(wandb.Api().runs(
    f"{WANDB_ENTITY}/{WANDB_PROJECT}")) if Segmentation.SAM.sam_lora_util.EVALUATED_TAG in run.tags and run.state == "finished"
    and run.name in models]

for run in runs:
    artifact_name = f"{run.name.lower()}_{params_artifact_name}:v0"
    print(f"Evaluating run {run.name} (artifact {artifact_name})")

    run_artifacts = [a for a in list(
        run.logged_artifacts()) if a.type == "model"]
    artifact = next(
        (a for a in run_artifacts if a.name == artifact_name), None)
    if artifact is None:
        raise ValueError(
            f"No artifact of type 'model' found with name '{artifact_name}'")

    params, _ = Segmentation.SAM.sam_lora_util.get_params_from_artifact(
        artifact, device)
    model = Segmentation.SAM.sam_lora_util.initialize_sam_lora_with_params(
        run.config, params, device)
    model.eval()

    # Use same test dirs derived from the run's config
    model_test_img_dir = Path(
        DATASETS_DIR) / run.config["dataset"] / "test" / HIGHRES_IMG_PATCHES_DIR_NAME
    model_test_mask_dir = Path(
        DATASETS_DIR) / run.config["dataset"] / "test" / HIGHRES_MASK_PATCHES_DIR_NAME

    dice_scores, iou_scores, clDice_scores, tprec_scores, tsens_scores = [], [], [], [], []
    pp_dice_scores, pp_iou_scores, pp_clDice_scores, pp_tprec_scores, pp_tsens_scores = [], [], [], [], []

    for img_path in model_test_img_dir.glob("*.png"):
        groundtruth_mask_path = model_test_mask_dir / img_path.name

        img = load_transform_image(img_path)
        input_list = Segmentation.SAM.sam_lora_util.get_batched_input_list(
            img.unsqueeze(0).to(device))
        output = model(batched_input=input_list, multimask_output=False)

        output_mask = output[0]['masks'].detach().cpu()

        # "raw" model output
        groundtruth_mask = load_transform_image(
            groundtruth_mask_path, is_mask=True).detach().cpu()
        dice = Segmentation.SAM.sam_lora_util.hard_dice_score(
            output_mask, groundtruth_mask)
        iou = Segmentation.SAM.sam_lora_util.iou_score(
            output_mask, groundtruth_mask)
        output_mask_np = output_mask.squeeze(0).cpu().numpy()
        groundtruth_mask_np = groundtruth_mask.squeeze(0).cpu().numpy()
        clDice, tprec, tsens = Segmentation.SAM.sam_lora_util.hard_clDice(
            output_mask_np, groundtruth_mask_np)

        dice_scores.append(dice)
        iou_scores.append(iou)
        clDice_scores.append(clDice)
        tprec_scores.append(tprec)
        tsens_scores.append(tsens)

        # post-processed (small objects removed) model output
        postprocessed_output_mask = Segmentation.Postprocessing.segmentation_postprocessing.remove_small_objects_from_batch(
            output_mask)
        pp_dice = Segmentation.SAM.sam_lora_util.hard_dice_score(
            postprocessed_output_mask, groundtruth_mask)
        pp_iou = Segmentation.SAM.sam_lora_util.iou_score(
            postprocessed_output_mask, groundtruth_mask)
        postprocessed_output_mask_np = postprocessed_output_mask.squeeze(
            0).cpu().numpy()
        pp_clDice, pp_tprec, pp_tsens = Segmentation.SAM.sam_lora_util.hard_clDice(
            postprocessed_output_mask_np, groundtruth_mask_np)

        pp_dice_scores.append(pp_dice)
        pp_iou_scores.append(pp_iou)
        pp_clDice_scores.append(pp_clDice)
        pp_tprec_scores.append(pp_tprec)
        pp_tsens_scores.append(pp_tsens)

    results[run.name] = {
        "Dice": np.mean(dice_scores),
        "IoU": np.mean(iou_scores),
        "clDice": np.mean(clDice_scores),
        "tprec": np.mean(tprec_scores),
        "tsens": np.mean(tsens_scores),

        "Dice Postprocessed": np.mean(pp_dice_scores),
        "IoU Postprocessed": np.mean(pp_iou_scores),
        "clDice Postprocessed": np.mean(pp_clDice_scores),
        "tprec Postprocessed": np.mean(pp_tprec_scores),
        "tsens Postprocessed": np.mean(pp_tsens_scores),
    }

    for k in loss_config_keys:
        results[run.name][k] = run.config.get(k, 0.0)

    del model, params
    torch.cuda.empty_cache()

df = pd.DataFrame(results).T
df.index.name = "Model"

best_per_metric = df.idxmax()


def bold_best(val, model_name, metric):
    s = f"{val:.4f}"
    return f"\\textbf{{{s}}}" if model_name == best_per_metric[metric] else s


styled_df = df.copy().astype(str)
for col in df.columns:
    for model_name in df.index:
        if col in loss_config_keys:
            # don't display loss columns in bold
            styled_df.loc[model_name, col] = f"{df.loc[model_name, col]:.2f}"
        else:
            styled_df.loc[model_name, col] = bold_best(
                df.loc[model_name, col], model_name, col)


def highlight_best(s):
    if s.name in loss_config_keys:
        # no highlighting for loss columns
        return ''
    is_best = s == s.max()
    return ['font-weight: bold' if v else '' for v in is_best]


print("\n\nModel Comparison (mean over test images, small objects removed):")
print("=" * 80)
df.style.apply(highlight_best, axis=0).format("{:.4f}")

Evaluating run SAM_LoRA_Finetuning_20260212_155155 (artifact sam_lora_finetuning_20260212_155155_params_minloss:v0)


wandb:   1 of 1 files downloaded.  


Evaluating run SAM_LoRA_Finetuning_20260212_155412 (artifact sam_lora_finetuning_20260212_155412_params_minloss:v0)


wandb:   1 of 1 files downloaded.  


Evaluating run SAM_LoRA_Finetuning_20260212_155506 (artifact sam_lora_finetuning_20260212_155506_params_minloss:v0)


...

wandb:   1 of 1 files downloaded.  


Evaluating run SAM_LoRA_Finetuning_20260212_163023 (artifact sam_lora_finetuning_20260212_163023_params_minloss:v0)


wandb:   1 of 1 files downloaded.  




Model Comparison (mean over test images, small objects removed):
                                                Dice              IoU           clDice            tprec            tsens bce_loss_weight cl_dice_loss_weight dice_loss_weight focal_loss_weight junction_heatmap_weight_scale junction_patch_weight skeleton_recall_loss_weight topological_loss_weight
Model                                                                                                                                                                                                                                                                                                  
SAM_LoRA_Finetuning_20260212_155155           0.8006           0.6695           0.9316           0.9804           0.8914          0.5000              0.0000           0.0000            0.0000                        0.0000                0.0000                      0.0000                  0.5000
SAM_LoRA_Finetuning_20260212_155412          

,Dice,IoU,clDice,tprec,tsens,bce_loss_weight,cl_dice_loss_weight,dice_loss_weight,focal_loss_weight,junction_heatmap_weight_scale,junction_patch_weight,skeleton_recall_loss_weight,topological_loss_weight
Model,,,,,,,,,,,,,
SAM_LoRA_Finetuning_20260212_155155,0.8006,0.6695,0.9316,0.9804,0.8914,0.5000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.5000
SAM_LoRA_Finetuning_20260212_155412,0.8214,0.6987,0.9477,0.9832,0.9174,0.5000,0.5000,0.0000,0.0000,0.0000,0.0000,0.0000,0.5000
SAM_LoRA_Finetuning_20260212_155506,0.8385,0.7238,0.9567,0.9877,0.9298,0.2500,0.5000,0.5000,0.0000,0.0000,0.0000,0.0000,0.0000
SAM_LoRA_Finetuning_20260212_155528,0.8531,0.7452,0.9594,0.9640,0.9565,0.2500,0.0000,0.5000,0.0000,0.0000,0.0000,0.5000,0.0000
SAM_LoRA_Finetuning_20260212_155556,0.8369,0.7214,0.9516,0.9738,0.9334,0.2500,0.5000,0.5000,0.0000,1000.0000,0.0000,0.0000,0.0000
SAM_LoRA_Finetuning_20260212_161635,0.8355,0.7194,0.9547,0.9878,0.9262,0.0000,0.5000,0.5000,0.2500,0.0000,0.0000,0.0000,0.0000
SAM_LoRA_Finetuning_20260212_163023,0.8416,0.7282,0.9566,0.9819,0.9347,0.2500,0.5000,0.5000,0.0000,250.0000,0.0000,0.0000,0.0000
